# Siamese Networks: Learning Similarity

## 1. Introduction

**Siamese Networks** are a powerful neural network architecture designed to learn similarity between inputs. Unlike traditional classification networks that predict labels, Siamese networks learn to determine whether two inputs are similar or dissimilar.

### Why Siamese Networks Matter

Siamese networks are foundational for:
- **One-shot learning**: Recognizing new classes from just a single example
- **Face verification**: "Is this the same person?"
- **Signature verification**: Detecting forgeries
- **Similarity search**: Finding similar images, products, or documents
- **Metric learning**: Learning meaningful distance functions in embedding space

### What We'll Build

In this notebook, we'll:
1. Understand similarity metrics and distance functions
2. Implement **contrastive loss** to learn discriminative embeddings
3. Build a Siamese network with **shared weights**
4. Train on MNIST digit pairs to learn similarity
5. Visualize learned embeddings using t-SNE
6. Explore **triplet loss** for more sophisticated training
7. Demonstrate **one-shot learning** capabilities

### Key Intuitions

- **Shared weights**: Two identical networks process different inputs, ensuring consistent representations
- **Contrastive learning**: Pull similar pairs together, push dissimilar pairs apart
- **Embedding space**: Learn a space where distance corresponds to semantic similarity

## 2. Setup

Let's import the necessary libraries and configure our environment.

In [ ]:
# Standard library imports
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List

# PyTorch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import datasets, transforms

# Scikit-learn for visualization
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import euclidean_distances

# Shared utilities
from aiml_notebooks import get_device, set_seed

# Enable autoreload for development
%load_ext autoreload
%autoreload 2

Set random seed for reproducibility and configure device.

In [ ]:
set_seed(42)
device = get_device()

Configure matplotlib for better visualizations.

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 3. Understanding Similarity Metrics

Before building Siamese networks, we need to understand how to measure similarity between vectors. Two common metrics are **Euclidean distance** and **cosine similarity**.

### Euclidean Distance (L2 Distance)

The Euclidean distance measures the straight-line distance between two points:

$$d(x_1, x_2) = \|x_1 - x_2\|_2 = \sqrt{\sum_{i=1}^n (x_{1,i} - x_{2,i})^2}$$

- **Lower distance** = more similar
- **Higher distance** = less similar

### Cosine Similarity

Cosine similarity measures the angle between two vectors:

$$\text{cosine\_sim}(x_1, x_2) = \frac{x_1 \cdot x_2}{\|x_1\| \|x_2\|}$$

- **1.0** = identical direction (very similar)
- **0.0** = orthogonal (unrelated)
- **-1.0** = opposite direction (dissimilar)

Let's implement both distance metrics.

In [ ]:
def euclidean_distance(x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
    """
    Compute Euclidean distance between two batches of vectors.
    
    Args:
        x1: Tensor of shape (batch_size, embedding_dim)
        x2: Tensor of shape (batch_size, embedding_dim)
    
    Returns:
        distances: Tensor of shape (batch_size,)
    """
    return torch.sqrt(torch.sum((x1 - x2) ** 2, dim=1))

def cosine_similarity(x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
    """
    Compute cosine similarity between two batches of vectors.
    
    Args:
        x1: Tensor of shape (batch_size, embedding_dim)
        x2: Tensor of shape (batch_size, embedding_dim)
    
    Returns:
        similarities: Tensor of shape (batch_size,)
    """
    return F.cosine_similarity(x1, x2, dim=1)

Test our distance functions with simple examples.

In [ ]:
# Create test vectors
v1 = torch.tensor([[1.0, 0.0, 0.0]])
v2 = torch.tensor([[1.0, 0.0, 0.0]])  # Identical
v3 = torch.tensor([[0.0, 1.0, 0.0]])  # Orthogonal
v4 = torch.tensor([[-1.0, 0.0, 0.0]]) # Opposite

print("Euclidean Distance Tests:")
print(f"  Identical vectors: {euclidean_distance(v1, v2).item():.3f}")
print(f"  Orthogonal vectors: {euclidean_distance(v1, v3).item():.3f}")
print(f"  Opposite vectors: {euclidean_distance(v1, v4).item():.3f}")

print("\nCosine Similarity Tests:")
print(f"  Identical vectors: {cosine_similarity(v1, v2).item():.3f}")
print(f"  Orthogonal vectors: {cosine_similarity(v1, v3).item():.3f}")
print(f"  Opposite vectors: {cosine_similarity(v1, v4).item():.3f}")

**Key insight**: Euclidean distance is sensitive to magnitude, while cosine similarity only cares about direction. For Siamese networks, we typically use Euclidean distance on **normalized embeddings** or cosine similarity.

## 4. Contrastive Loss: The Heart of Siamese Networks

**Contrastive loss** is the key innovation that makes Siamese networks work. It has two goals:

1. **Similar pairs (label=1)**: Minimize the distance → pull embeddings together
2. **Dissimilar pairs (label=0)**: Maximize the distance (up to a margin) → push embeddings apart

The mathematical formulation:

$$\mathcal{L} = \frac{1}{2N} \sum_{i=1}^N \left[ y_i \cdot d_i^2 + (1-y_i) \cdot \max(0, m - d_i)^2 \right]$$

Where:
- $d_i$ = Euclidean distance between embeddings
- $y_i$ = 1 for similar pairs, 0 for dissimilar pairs
- $m$ = margin (typical value: 1.0 or 2.0)

### Intuition

- **Similar pairs**: Loss = $d^2$. Want distance = 0.
- **Dissimilar pairs**: Loss = $\max(0, m - d)^2$. Want distance ≥ margin. No penalty if already far apart.

Implement the contrastive loss function.

In [ ]:
class ContrastiveLoss(nn.Module):
    """
    Contrastive loss function for Siamese networks.
    
    Args:
        margin: Minimum distance for dissimilar pairs
    """
    def __init__(self, margin: float = 2.0):
        super().__init__()
        self.margin = margin
    
    def forward(self, output1: torch.Tensor, output2: torch.Tensor, label: torch.Tensor) -> torch.Tensor:
        """
        Args:
            output1: Embeddings from first input (batch_size, embedding_dim)
            output2: Embeddings from second input (batch_size, embedding_dim)
            label: 1 for similar pairs, 0 for dissimilar pairs (batch_size,)
        
        Returns:
            loss: Scalar loss value
        """
        # Compute Euclidean distance
        distance = euclidean_distance(output1, output2)
        
        # Contrastive loss formula
        loss_similar = label * distance.pow(2)
        loss_dissimilar = (1 - label) * F.relu(self.margin - distance).pow(2)
        
        loss = torch.mean(loss_similar + loss_dissimilar)
        return loss

Visualize how contrastive loss behaves for different distances.

In [ ]:
# Create range of distances
distances = torch.linspace(0, 3, 100)
margin = 2.0

# Compute losses for similar and dissimilar pairs
loss_similar = distances.pow(2)
loss_dissimilar = F.relu(margin - distances).pow(2)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Similar pairs
ax1.plot(distances.numpy(), loss_similar.numpy(), 'b-', linewidth=2)
ax1.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax1.set_xlabel('Distance between embeddings')
ax1.set_ylabel('Loss')
ax1.set_title('Similar Pairs (label=1)\nWant distance → 0')
ax1.grid(True, alpha=0.3)

# Dissimilar pairs
ax2.plot(distances.numpy(), loss_dissimilar.numpy(), 'r-', linewidth=2)
ax2.axvline(x=margin, color='g', linestyle='--', linewidth=2, label=f'Margin = {margin}')
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax2.set_xlabel('Distance between embeddings')
ax2.set_ylabel('Loss')
ax2.set_title('Dissimilar Pairs (label=0)\nWant distance ≥ margin')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key observations**:
- Similar pairs: Loss increases quadratically with distance → strong pressure to bring embeddings close
- Dissimilar pairs: Loss is zero beyond the margin → no wasted effort pushing already-separated pairs further apart
- The margin parameter controls how far apart dissimilar pairs should be

## 5. Building a Siamese Network

A **Siamese network** consists of:
1. **Twin networks** with **shared weights** that process each input
2. An **embedding network** that maps inputs to a fixed-dimensional space
3. A **distance function** to compare embeddings

The key insight: **shared weights** ensure that both inputs are processed identically, so differences in embeddings reflect differences in the inputs, not the network.

Define a simple convolutional embedding network for MNIST digits.

In [ ]:
class EmbeddingNetwork(nn.Module):
    """
    Convolutional network that maps images to embeddings.
    
    Args:
        embedding_dim: Dimension of output embedding vector
    """
    def __init__(self, embedding_dim: int = 128):
        super().__init__()
        
        # Convolutional feature extractor
        self.conv = nn.Sequential(
            # 28x28x1 -> 24x24x32
            nn.Conv2d(1, 32, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2),  # -> 12x12x32
            
            # 12x12x32 -> 8x8x64
            nn.Conv2d(32, 64, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2),  # -> 4x4x64
        )
        
        # Fully connected layers to embedding
        self.fc = nn.Sequential(
            nn.Linear(4 * 4 * 64, 256),
            nn.ReLU(),
            nn.Linear(256, embedding_dim)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input images (batch_size, 1, 28, 28)
        
        Returns:
            embeddings: (batch_size, embedding_dim)
        """
        x = self.conv(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc(x)
        return x

Build the complete Siamese network.

In [ ]:
class SiameseNetwork(nn.Module):
    """
    Siamese Network with shared embedding network.
    
    Args:
        embedding_dim: Dimension of embedding space
    """
    def __init__(self, embedding_dim: int = 128):
        super().__init__()
        # Single embedding network (shared weights for both inputs)
        self.embedding_net = EmbeddingNetwork(embedding_dim)
    
    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Process two inputs through the same embedding network.
        
        Args:
            x1: First input images (batch_size, 1, 28, 28)
            x2: Second input images (batch_size, 1, 28, 28)
        
        Returns:
            (output1, output2): Tuple of embeddings, each (batch_size, embedding_dim)
        """
        # Pass both inputs through the SAME network (shared weights)
        output1 = self.embedding_net(x1)
        output2 = self.embedding_net(x2)
        return output1, output2
    
    def get_embedding(self, x: torch.Tensor) -> torch.Tensor:
        """
        Get embedding for a single input.
        
        Args:
            x: Input images (batch_size, 1, 28, 28)
        
        Returns:
            embeddings: (batch_size, embedding_dim)
        """
        return self.embedding_net(x)

Create an instance and verify the architecture.

In [ ]:
model = SiameseNetwork(embedding_dim=128).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# Test forward pass
x1 = torch.randn(4, 1, 28, 28).to(device)
x2 = torch.randn(4, 1, 28, 28).to(device)
out1, out2 = model(x1, x2)

print(f"\nInput shape: {x1.shape}")
print(f"Output embedding shape: {out1.shape}")
print(f"Embedding dimension: {out1.shape[1]}")

**Key point**: Both inputs go through the **same** `embedding_net` with **shared weights**. This is what makes it "Siamese" - like identical twins!

## 6. Creating MNIST Pair Dataset

To train a Siamese network, we need pairs of images with labels indicating whether they're similar (same digit) or dissimilar (different digits).

Our dataset will generate:
- **Positive pairs**: Two images of the same digit (label=1)
- **Negative pairs**: Two images of different digits (label=0)

Load MNIST dataset.

In [ ]:
# Transform: convert to tensor and normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

# Load training and test sets
mnist_train = datasets.MNIST('./data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST('./data', train=False, download=True, transform=transform)

print(f"Training samples: {len(mnist_train)}")
print(f"Test samples: {len(mnist_test)}")

Create a custom dataset that generates pairs.

In [ ]:
class MNISTPairDataset(Dataset):
    """
    Dataset that generates pairs of MNIST images.
    
    Args:
        mnist_dataset: Original MNIST dataset
        num_pairs: Number of pairs to generate
        positive_ratio: Ratio of positive (similar) pairs
    """
    def __init__(self, mnist_dataset, num_pairs: int = 10000, positive_ratio: float = 0.5):
        self.mnist_dataset = mnist_dataset
        self.num_pairs = num_pairs
        self.positive_ratio = positive_ratio
        
        # Organize images by label for efficient sampling
        self.label_to_indices = {i: [] for i in range(10)}
        for idx, (_, label) in enumerate(mnist_dataset):
            self.label_to_indices[label].append(idx)
    
    def __len__(self):
        return self.num_pairs
    
    def __getitem__(self, idx):
        """
        Returns:
            (img1, img2, label): Tuple of two images and similarity label (1=similar, 0=dissimilar)
        """
        # Decide whether to create positive or negative pair
        should_get_same_class = np.random.rand() < self.positive_ratio
        
        if should_get_same_class:
            # Positive pair: same digit
            label_class = np.random.randint(0, 10)
            idx1, idx2 = np.random.choice(self.label_to_indices[label_class], size=2, replace=False)
            label = 1.0
        else:
            # Negative pair: different digits
            label1, label2 = np.random.choice(10, size=2, replace=False)
            idx1 = np.random.choice(self.label_to_indices[label1])
            idx2 = np.random.choice(self.label_to_indices[label2])
            label = 0.0
        
        img1, _ = self.mnist_dataset[idx1]
        img2, _ = self.mnist_dataset[idx2]
        
        return img1, img2, torch.tensor(label, dtype=torch.float32)

Create train and test pair datasets.

In [ ]:
# Create pair datasets
train_pairs = MNISTPairDataset(mnist_train, num_pairs=20000, positive_ratio=0.5)
test_pairs = MNISTPairDataset(mnist_test, num_pairs=5000, positive_ratio=0.5)

# Create dataloaders
batch_size = 64
train_loader = DataLoader(train_pairs, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_pairs, batch_size=batch_size, shuffle=False)

print(f"Training pairs: {len(train_pairs)}")
print(f"Test pairs: {len(test_pairs)}")
print(f"Batches per epoch: {len(train_loader)}")

Visualize some example pairs.

In [ ]:
def visualize_pairs(pairs_dataset, num_pairs: int = 4):
    """
    Visualize pairs of images with their labels.
    """
    fig, axes = plt.subplots(num_pairs, 2, figsize=(6, num_pairs * 2))
    
    for i in range(num_pairs):
        img1, img2, label = pairs_dataset[i]
        
        # Denormalize for visualization
        img1 = img1.squeeze().numpy() * 0.3081 + 0.1307
        img2 = img2.squeeze().numpy() * 0.3081 + 0.1307
        
        axes[i, 0].imshow(img1, cmap='gray')
        axes[i, 0].axis('off')
        axes[i, 1].imshow(img2, cmap='gray')
        axes[i, 1].axis('off')
        
        pair_type = "Similar" if label == 1.0 else "Dissimilar"
        color = 'green' if label == 1.0 else 'red'
        axes[i, 0].set_title(f"{pair_type} Pair", color=color, fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

visualize_pairs(train_pairs, num_pairs=6)

## 7. Training the Siamese Network

Now we'll train the Siamese network using contrastive loss. The network will learn to:
- Map similar digits close together in embedding space
- Map dissimilar digits far apart

Set up training components.

In [ ]:
# Create fresh model
model = SiameseNetwork(embedding_dim=128).to(device)

# Loss and optimizer
criterion = ContrastiveLoss(margin=2.0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Training setup complete!")

Define training and evaluation functions.

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """
    Train for one epoch.
    """
    model.train()
    total_loss = 0
    
    for img1, img2, labels in dataloader:
        img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)
        
        # Forward pass
        output1, output2 = model(img1, img2)
        
        # Compute loss
        loss = criterion(output1, output2, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def evaluate(model, dataloader, criterion, device):
    """
    Evaluate on validation/test set.
    """
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for img1, img2, labels in dataloader:
            img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)
            
            # Forward pass
            output1, output2 = model(img1, img2)
            
            # Compute loss
            loss = criterion(output1, output2, labels)
            total_loss += loss.item()
            
            # Compute accuracy (threshold = margin / 2)
            distances = euclidean_distance(output1, output2)
            predictions = (distances < 1.0).float()  # Predict similar if distance < threshold
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy

Train the model.

In [ ]:
# Training loop
num_epochs = 10
train_losses = []
test_losses = []
test_accuracies = []

for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs}:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

Visualize training progress.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(train_losses, label='Train Loss', marker='o')
ax1.plot(test_losses, label='Test Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Progress - Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curve
ax2.plot(test_accuracies, label='Test Accuracy', marker='o', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Test Accuracy')
ax2.set_ylim([0, 1])
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Test Accuracy: {test_accuracies[-1]:.4f}")

**What's happening**: The network is learning to produce embeddings where similar digits are close together and dissimilar digits are far apart!

## 8. Visualizing Learned Embeddings

Let's visualize what the network has learned by extracting embeddings for MNIST test images and projecting them to 2D using t-SNE.

Extract embeddings for all test images.

In [ ]:
def extract_embeddings(model, dataset, num_samples: int = 1000):
    """
    Extract embeddings for a subset of images.
    """
    model.eval()
    embeddings = []
    labels = []
    
    # Sample random indices
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    
    with torch.no_grad():
        for idx in indices:
            img, label = dataset[idx]
            img = img.unsqueeze(0).to(device)  # Add batch dimension
            
            embedding = model.get_embedding(img)
            embeddings.append(embedding.cpu().numpy())
            labels.append(label)
    
    embeddings = np.vstack(embeddings)
    labels = np.array(labels)
    return embeddings, labels

# Extract embeddings
print("Extracting embeddings...")
embeddings, labels = extract_embeddings(model, mnist_test, num_samples=1000)
print(f"Embeddings shape: {embeddings.shape}")

Use t-SNE to project embeddings to 2D for visualization.

In [ ]:
print("Running t-SNE...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d = tsne.fit_transform(embeddings)
print("t-SNE complete!")

Visualize the embedding space.

In [ ]:
plt.figure(figsize=(12, 10))

# Plot each digit class with different color
colors = plt.cm.tab10(np.linspace(0, 1, 10))
for digit in range(10):
    mask = labels == digit
    plt.scatter(
        embeddings_2d[mask, 0],
        embeddings_2d[mask, 1],
        c=[colors[digit]],
        label=f'Digit {digit}',
        alpha=0.6,
        s=50
    )

plt.legend(loc='best', fontsize=10)
plt.title('t-SNE Visualization of Learned Embeddings', fontsize=14, fontweight='bold')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Observe**: 
- Digits of the same class cluster together
- Different digits are separated in the embedding space
- The network learned semantic similarity without being explicitly trained to classify!
- Similar-looking digits (like 4 and 9) may be closer than very different digits (like 0 and 1)

## 9. Testing Similarity on New Pairs

Let's test our trained network on some example pairs to see how well it judges similarity.

Create a function to visualize similarity predictions.

In [ ]:
def test_similarity(model, pairs_dataset, num_pairs: int = 6):
    """
    Test the model on pairs and visualize predictions.
    """
    model.eval()
    
    fig, axes = plt.subplots(num_pairs, 3, figsize=(10, num_pairs * 2))
    
    with torch.no_grad():
        for i in range(num_pairs):
            img1, img2, label = pairs_dataset[i]
            
            # Get embeddings
            img1_batch = img1.unsqueeze(0).to(device)
            img2_batch = img2.unsqueeze(0).to(device)
            out1, out2 = model(img1_batch, img2_batch)
            
            # Compute distance
            distance = euclidean_distance(out1, out2).item()
            prediction = "Similar" if distance < 1.0 else "Dissimilar"
            ground_truth = "Similar" if label == 1.0 else "Dissimilar"
            
            # Denormalize for visualization
            img1_vis = img1.squeeze().numpy() * 0.3081 + 0.1307
            img2_vis = img2.squeeze().numpy() * 0.3081 + 0.1307
            
            # Plot images
            axes[i, 0].imshow(img1_vis, cmap='gray')
            axes[i, 0].axis('off')
            axes[i, 0].set_title('Image 1')
            
            axes[i, 1].imshow(img2_vis, cmap='gray')
            axes[i, 1].axis('off')
            axes[i, 1].set_title('Image 2')
            
            # Plot prediction
            axes[i, 2].axis('off')
            correct = (prediction == ground_truth)
            color = 'green' if correct else 'red'
            
            text = f"Distance: {distance:.2f}\n"
            text += f"Prediction: {prediction}\n"
            text += f"Ground Truth: {ground_truth}\n"
            text += f"Correct: {correct}"
            
            axes[i, 2].text(
                0.5, 0.5, text,
                ha='center', va='center',
                fontsize=11,
                bbox=dict(boxstyle='round', facecolor=color, alpha=0.3)
            )
    
    plt.tight_layout()
    plt.show()

test_similarity(model, test_pairs, num_pairs=8)

## 10. Triplet Loss: An Advanced Alternative

**Triplet loss** is an alternative to contrastive loss that works with triplets instead of pairs:

- **Anchor**: Reference image
- **Positive**: Different image of the same class (should be close)
- **Negative**: Image of a different class (should be far)

The loss encourages:
$$d(\text{anchor}, \text{positive}) + \text{margin} < d(\text{anchor}, \text{negative})$$

Mathematical formulation:
$$\mathcal{L} = \max(0, d(a, p) - d(a, n) + m)$$

Where:
- $d(a, p)$ = distance between anchor and positive
- $d(a, n)$ = distance between anchor and negative
- $m$ = margin

Implement triplet loss.

In [ ]:
class TripletLoss(nn.Module):
    """
    Triplet loss function.
    
    Args:
        margin: Minimum difference between positive and negative distances
    """
    def __init__(self, margin: float = 1.0):
        super().__init__()
        self.margin = margin
    
    def forward(self, anchor: torch.Tensor, positive: torch.Tensor, negative: torch.Tensor) -> torch.Tensor:
        """
        Args:
            anchor: Anchor embeddings (batch_size, embedding_dim)
            positive: Positive embeddings (batch_size, embedding_dim)
            negative: Negative embeddings (batch_size, embedding_dim)
        
        Returns:
            loss: Scalar loss value
        """
        # Compute distances
        dist_pos = euclidean_distance(anchor, positive)
        dist_neg = euclidean_distance(anchor, negative)
        
        # Triplet loss: want dist_pos + margin < dist_neg
        loss = F.relu(dist_pos - dist_neg + self.margin)
        return torch.mean(loss)

Visualize how triplet loss behaves.

In [ ]:
# Fix positive distance at different values
dist_neg_range = torch.linspace(0, 3, 100)
margin = 1.0

fig, ax = plt.subplots(figsize=(10, 6))

for dist_pos in [0.2, 0.5, 1.0, 1.5]:
    loss = F.relu(dist_pos - dist_neg_range + margin)
    ax.plot(dist_neg_range.numpy(), loss.numpy(), label=f'dist(A,P)={dist_pos}', linewidth=2)

ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax.set_xlabel('Distance(Anchor, Negative)')
ax.set_ylabel('Loss')
ax.set_title(f'Triplet Loss Behavior (margin={margin})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Key insight: Loss is zero when negative distance exceeds positive distance by the margin.")
print("This encourages: dist(anchor, positive) + margin < dist(anchor, negative)")

**Triplet loss advantages**:
- More efficient learning: directly optimizes relative distances
- Better for one-shot learning tasks
- Used in FaceNet and many modern face recognition systems

**Challenge**: Requires careful **triplet mining** - selecting informative hard triplets for training.

## 11. One-Shot Learning Demonstration

**One-shot learning** is the ability to learn from just a single example. Siamese networks excel at this!

Scenario: Given one example of a new digit, can we recognize other instances?

Approach:
1. **Support set**: One example of each class
2. **Query**: New image to classify
3. **Classification**: Find the support example with the smallest distance to the query

Implement one-shot classification.

In [ ]:
def one_shot_classify(model, query_img, support_set, support_labels, device):
    """
    Classify a query image using one-shot learning.
    
    Args:
        model: Trained Siamese network
        query_img: Query image tensor (1, 28, 28)
        support_set: List of support images, one per class
        support_labels: Labels for support images
        device: Device for computation
    
    Returns:
        predicted_label: Predicted class
        distances: Distance to each support example
    """
    model.eval()
    
    with torch.no_grad():
        # Get query embedding
        query_embedding = model.get_embedding(query_img.unsqueeze(0).to(device))
        
        # Compute distances to all support examples
        distances = []
        for support_img in support_set:
            support_embedding = model.get_embedding(support_img.unsqueeze(0).to(device))
            dist = euclidean_distance(query_embedding, support_embedding).item()
            distances.append(dist)
        
        # Predict: closest support example
        distances = np.array(distances)
        predicted_idx = np.argmin(distances)
        predicted_label = support_labels[predicted_idx]
    
    return predicted_label, distances

Create a support set with one example per digit.

In [ ]:
# Create support set: one example per digit
support_set = []
support_labels = []

for digit in range(10):
    # Find first occurrence of this digit
    for idx in range(len(mnist_test)):
        img, label = mnist_test[idx]
        if label == digit:
            support_set.append(img)
            support_labels.append(label)
            break

print(f"Support set created with {len(support_set)} examples (one per digit)")

Visualize the support set.

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, (img, label) in enumerate(zip(support_set, support_labels)):
    img_vis = img.squeeze().numpy() * 0.3081 + 0.1307
    axes[i].imshow(img_vis, cmap='gray')
    axes[i].set_title(f'Digit {label}')
    axes[i].axis('off')
plt.suptitle('Support Set (One Example Per Class)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Test one-shot classification on random query images.

In [ ]:
# Test on random queries
num_queries = 8
fig, axes = plt.subplots(num_queries, 11, figsize=(16, num_queries * 1.5))

correct = 0
for i in range(num_queries):
    # Random query image
    idx = np.random.randint(len(mnist_test))
    query_img, true_label = mnist_test[idx]
    
    # Classify
    predicted_label, distances = one_shot_classify(model, query_img, support_set, support_labels, device)
    
    # Visualize query
    query_vis = query_img.squeeze().numpy() * 0.3081 + 0.1307
    axes[i, 0].imshow(query_vis, cmap='gray')
    axes[i, 0].set_title(f'Query\nTrue: {true_label}')
    axes[i, 0].axis('off')
    
    # Visualize distances to each support example
    for j in range(10):
        support_vis = support_set[j].squeeze().numpy() * 0.3081 + 0.1307
        axes[i, j+1].imshow(support_vis, cmap='gray')
        
        # Highlight closest match
        if j == predicted_label:
            color = 'green' if predicted_label == true_label else 'red'
            for spine in axes[i, j+1].spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(3)
        
        axes[i, j+1].set_title(f'd={distances[j]:.2f}', fontsize=8)
        axes[i, j+1].axis('off')
    
    if predicted_label == true_label:
        correct += 1

plt.suptitle(f'One-Shot Classification (Accuracy: {correct}/{num_queries})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Remarkable**: The network can classify digits by comparing to just **one example** of each class! This is the power of learned similarity.

Evaluate one-shot accuracy on larger test set.

In [ ]:
# Evaluate on 1000 random test images
num_test = 1000
correct = 0

for _ in range(num_test):
    idx = np.random.randint(len(mnist_test))
    query_img, true_label = mnist_test[idx]
    predicted_label, _ = one_shot_classify(model, query_img, support_set, support_labels, device)
    
    if predicted_label == true_label:
        correct += 1

accuracy = correct / num_test
print(f"One-Shot Learning Accuracy: {accuracy:.4f} ({correct}/{num_test})")
print(f"\nThis is impressive considering we only showed ONE example of each digit!")

## 12. Key Takeaways

### Core Concepts

1. **Siamese Networks Learn Similarity**
   - Instead of classifying, they learn whether inputs are similar or dissimilar
   - Powerful for tasks where labeled data is scarce

2. **Shared Weights Are Critical**
   - Both inputs pass through the **same** network with **identical weights**
   - Ensures differences in embeddings reflect differences in inputs, not processing

3. **Contrastive Loss Pulls and Pushes**
   - Similar pairs: minimize distance (pull together)
   - Dissimilar pairs: maximize distance up to margin (push apart)
   - The margin prevents wasted effort on already-separated pairs

4. **Embedding Space Captures Semantics**
   - Network learns to map inputs to a space where distance = semantic similarity
   - Similar objects cluster together, dissimilar objects separate

5. **One-Shot Learning**
   - Can recognize new classes from just one example
   - Classification becomes nearest-neighbor search in embedding space

### Practical Applications

- **Face Verification**: "Is this the same person?" (FaceNet, DeepFace)
- **Signature Verification**: Detecting forgeries
- **Product Matching**: Finding similar products in e-commerce
- **Few-Shot Learning**: Learning from limited labeled data
- **Image Retrieval**: Finding similar images in large databases

### Alternative Approaches

- **Triplet Loss**: Uses anchor-positive-negative triplets instead of pairs
- **Quadruplet Loss**: Adds second negative for better discrimination
- **N-pair Loss**: Generalizes to multiple negatives per anchor

### Connection to Other Concepts

- **Metric Learning**: Siamese networks learn distance metrics
- **Contrastive Learning**: Self-supervised learning uses similar principles (SimCLR, MoCo)
- **Embeddings**: Foundation for recommendation systems, NLP (Word2Vec), and more

### What's Next?

After mastering Siamese networks, you're ready for:
- Contrastive learning for self-supervised pretraining
- Advanced metric learning techniques
- Few-shot and zero-shot learning
- Multi-modal embeddings (CLIP)

**Remember**: Siamese networks revolutionized how we think about similarity - instead of teaching a model categories, we teach it how to compare!

## 13. Exercises for Further Exploration

Try these experiments to deepen your understanding:

1. **Margin Sensitivity**: Retrain with different margins (0.5, 1.0, 2.0, 4.0). How does it affect clustering?

2. **Embedding Dimension**: Try embedding_dim = 32, 64, 256. Is higher always better?

3. **Different Datasets**: Apply to Fashion-MNIST or CIFAR-10. Does it work as well?

4. **Triplet vs Contrastive**: Implement training with triplet loss and compare results

5. **Hard Negative Mining**: Select pairs where the network is most uncertain for more efficient training

6. **Few-Shot Learning**: Modify one-shot to K-shot (use K examples per class instead of 1)

7. **Distance Metrics**: Compare Euclidean distance vs cosine similarity vs learnable distance functions